# SEC Proxy Ingestion Pipeline — ConnectOne Bancorp (CNOB) DEF 14A 2025


In [1]:
import os, sys, requests, json
from pathlib import Path
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from IPython.display import display
import pandas as pd
sys.path.insert(0, str(Path("..").resolve()))

from ingestion.metadata_model import DocumentMetadata
from ingestion.sec_html_parser import SECHTMLParser
from ingestion.sec_chunker import SECChunker

FILING_URL = "https://www.sec.gov/Archives/edgar/data/712771/000143774925011656/cnob20240411_def14a.htm"
SEC_USER_AGENT = os.getenv("SEC_USER_AGENT", "YaNa Research yashar@example.com")
DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
cache_path = DATA_DIR / "712771_000143774925011656.html"
if cache_path.exists():
    raw_html = cache_path.read_text(encoding="utf-8")
    print(f"Loaded from cache: {cache_path} ({len(raw_html):,} chars)")
else:
    retry = Retry(
        total=6,
        connect=6,
        read=6,
        backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        respect_retry_after_header=True,
    )
    session = requests.Session()
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    headers = {
        "User-Agent": SEC_USER_AGENT,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive",
    }
    print(f"Downloading from SEC EDGAR with User-Agent: {SEC_USER_AGENT}")
    resp = session.get(FILING_URL, headers=headers, timeout=45)
    resp.raise_for_status()
    raw_html = resp.text
    assert len(raw_html) > 10000, f"Unexpectedly short HTML payload: {len(raw_html)} chars"
    cache_path.write_text(raw_html, encoding="utf-8")
    print(f"Downloaded and cached: {cache_path} ({len(raw_html):,} chars)")


Loaded from cache: ../data/raw/712771_000143774925011656.html (899,270 chars)


## Filing Metadata


In [3]:
from datetime import date
metadata = DocumentMetadata(
    document_id="712771_000143774925011656",
    cik="712771",
    company_name="ConnectOne Bancorp, Inc.",
    form_type="DEF 14A",
    filing_date=date(2025, 4, 11),
    accession_number="0001437749-25-011656",
    source_url=FILING_URL,
    raw_html_path=str(cache_path),
)
print(metadata.model_dump_json(indent=2))


{
  "document_id": "712771_000143774925011656",
  "cik": "712771",
  "company_name": "ConnectOne Bancorp, Inc.",
  "form_type": "DEF 14A",
  "filing_date": "2025-04-11",
  "accession_number": "0001437749-25-011656",
  "source_url": "https://www.sec.gov/Archives/edgar/data/712771/000143774925011656/cnob20240411_def14a.htm",
  "fiscal_year_end": null,
  "raw_html_path": "../data/raw/712771_000143774925011656.html"
}


## Step 1: Parse HTML → Typed Blocks


In [4]:
parser = SECHTMLParser()
blocks = parser.parse(raw_html, metadata)
print(f"Total blocks extracted: {len(blocks)}")


/Users/yasharnaghdi/code/sec-rag-pipeline/ingestion/sec_html_parser.py:59: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(raw_html, "lxml")


Total blocks extracted: 579


In [5]:
from collections import Counter
type_counts = Counter(type(b).__name__ for b in blocks)
df_types = pd.DataFrame(type_counts.items(), columns=["BlockType", "Count"]).sort_values("Count", ascending=False)
display(df_types)


,BlockType,Count
3,TableBlock,207
2,ProseBlock,203
4,ImageBlock,88
1,HeadingBlock,69
0,XBRLTaggedBlock,10
5,FootnoteBlock,2


## Step 2: Inspect Sample Blocks


In [6]:
headings = [b for b in blocks if type(b).__name__ == "HeadingBlock"]
for h in headings[:5]:
    print(f"[{h.detection_method}] level={h.level} | section_id={h.section_id} | text={h.text[:80]}")


[bold_heuristic] level=2 | section_id=preamble | text=UNITED STATES
[bold_heuristic] level=2 | section_id=c46831e4022aed51fb717048379db10a10acec15d7a43bec1d027f833027bc7d | text=SECURITIES AND EXCHANGE COMMISSION
[bold_heuristic] level=2 | section_id=fe426eb7621caff62e0f03a7cda81ef147430f9e17db7a0c2a8f981245842476 | text=SCHEDULE 14A
[keyword_match] level=2 | section_id=446b38028c9c0ea0cbfe3fdabf23a7f00998eac4fb09e23405c09f4e73307802 | text=This Proxy Statement is being furnished to shareholders of ConnectOne Bancorp, I
[keyword_match] level=2 | section_id=c655779cace41cf9a158191d0c724583268382e80bc5b5c0c9d77edd6d9e597f | text=The accompanying proxy is solicited by the Board of Directors of ConnectOne Banc


In [7]:
tables = [b for b in blocks if type(b).__name__ == "TableBlock"]
print(f"Total tables: {len(tables)}")
if tables:
    t = tables[0]
    print(f"\nTable ID: {t.id}")
    print(f"Rows: {len(t.rows)}, Header rows: {t.header_row_count}, Merged cells: {t.has_merged_cells}")
    print(f"Footnotes: {t.footnotes}")
    print(f"\nLinearized (first 300 chars):\n{t.linearized_text[:300]}")


Total tables: 207

Table ID: 5db18f35b2193869df1a05b46dc7318250811690302e59c190c7c8fb498555ee
Rows: 1, Header rows: 0, Merged cells: False
Footnotes: {}

Linearized (first 300 chars):
Filed by the Registrant ☒ | Filed by a Party other than the Registrant ☐


In [8]:
xbrl_blocks = [b for b in blocks if type(b).__name__ == "XBRLTaggedBlock"]
print(f"XBRL-tagged blocks: {len(xbrl_blocks)}")
for x in xbrl_blocks[:3]:
    print(f"\n  text[:80]: {x.text[:80]}")
    for ann in x.xbrl_tags:
        print(f"  concept: {ann.concept_name}, contextRef: {ann.context_ref}")


XBRL-tagged blocks: 10

  text[:80]: false 0000712771 DEF 14A 0000712771 2024-01-01 2024-12-31 thunderdome:item 00007
  concept: dei:AmendmentFlag, contextRef: d_2024-01-01_2024-12-31
  concept: dei:EntityCentralIndexKey, contextRef: d_2024-01-01_2024-12-31
  concept: dei:DocumentType, contextRef: d_2024-01-01_2024-12-31

  text[:80]: UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 SCHEDULE
  concept: dei:EntityRegistrantName, contextRef: d_2024-01-01_2024-12-31
  concept: ecd:InsiderTrdPoliciesProcAdoptedFlag, contextRef: d_2024-01-01_2024-12-31
  concept: ecd:AwardTmgMnpiDiscTextBlock, contextRef: d_2024-01-01_2024-12-31
  concept: ecd:AwardTmgMethodTextBlock, contextRef: d_2024-01-01_2024-12-31
  concept: ecd:AwardTmgPredtrmndFlag, contextRef: d_2024-01-01_2024-12-31
  concept: ecd:MnpiDiscTimedForCompValFlag, contextRef: d_2024-01-01_2024-12-31
  concept: ecd:AwardTmgMnpiCnsdrdFlag, contextRef: d_2024-01-01_2024-12-31
  concept: ecd:PvpTableTextBlock, contex

## Step 3: Chunk Blocks


In [9]:
chunker = SECChunker()
chunks = chunker.chunk_blocks(blocks, metadata)
print(f"Total chunks: {len(chunks)}")


Total chunks: 639


In [10]:
section_counts = Counter(c.section_id for c in chunks)
df_sections = pd.DataFrame(section_counts.items(), columns=["SectionID", "ChunkCount"]).sort_values("ChunkCount", ascending=False)
display(df_sections.head(15))


,SectionID,ChunkCount
46,ba4383c064c8f0e49294a2e6ffb3f176ff68cda9329a82...,70
0,preamble,65
54,acbb270c5dfd35718eb5ba2fa90e2896f39c11cd017196...,49
3,446b38028c9c0ea0cbfe3fdabf23a7f00998eac4fb09e2...,49
53,71c8a52f5a822f80399c8f84a739c24e2b981ac137750f...,39
40,4f801202767a5b24646045e6a6df8b6b5dd819f8f4f7f6...,32
51,f5bbeeb20f0dcb74d868bba544e6faeacb6f2df79012cc...,24
20,6543df5391b2f8262213e327b7fb9911ed2a538bcbe3e9...,23
5,f100fd795adeb83777605de57b3f36131232903290d4fc...,20
15,d0114524a0757f140da3e3ace719be62a5a5be468596f1...,18


In [11]:
token_counts = [c.token_count for c in chunks]
prose_token_counts = [c.token_count for c in chunks if c.table_json is None]
print(f"Min tokens: {min(token_counts)}")
print(f"Max tokens: {max(token_counts)}")
print(f"Mean tokens: {sum(token_counts)/len(token_counts):.1f}")
assert max(prose_token_counts) <= 1200, f"CONSTRAINT VIOLATED: prose chunk has {max(prose_token_counts)} tokens"
print("✓ All prose chunks within 1200-token limit")


Min tokens: 2
Max tokens: 1197
Mean tokens: 172.4
✓ All prose chunks within 1200-token limit


In [12]:
for c in chunks[:3]:
    print(c.citation_string)


ConnectOne Bancorp, Inc. | DEF 14A | 2025-04-11 | preamble | chunk 0
ConnectOne Bancorp, Inc. | DEF 14A | 2025-04-11 | preamble | chunk 1
ConnectOne Bancorp, Inc. | DEF 14A | 2025-04-11 | preamble | chunk 2


## Step 4: Export to CSV


In [13]:
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
rows = [
    {
        "chunk_id": c.id,
        "chunk_index": c.chunk_index,
        "document_id": c.document_id,
        "section_id": c.section_id,
        "toc_page_range": f"{c.toc_page_range[0]}-{c.toc_page_range[1]}" if c.toc_page_range else "",
        "token_count": c.token_count,
        "citation_string": c.citation_string,
        "text_preview": c.text[:120].replace("\n", " "),
    }
    for c in chunks
]
df_chunks = pd.DataFrame(rows)
df_chunks.to_csv(OUTPUT_DIR / "chunks_cnob_m0.csv", index=False)
print(f"Exported {len(df_chunks)} chunks to output/chunks_cnob_m0.csv")
display(df_chunks.head(5))


Exported 639 chunks to output/chunks_cnob_m0.csv


,chunk_id,chunk_index,document_id,section_id,toc_page_range,token_count,citation_string,text_preview
0,f8554a6c-0cf4-4ed0-8ce8-115e2bec45ca,0,712771_000143774925011656,preamble,,810,"ConnectOne Bancorp, Inc. | DEF 14A | 2025-04-1...",false 0000712771 DEF 14A 0000712771 2024-01-01...
1,9e307006-ce28-4189-b5cf-5c07ff65c321,1,712771_000143774925011656,preamble,,678,"ConnectOne Bancorp, Inc. | DEF 14A | 2025-04-1...",UNITED STATES SECURITIES AND EXCHANGE COMMISSI...
2,420bbf52-c44d-4b23-a4e0-18c43c29b0c9,2,712771_000143774925011656,preamble,,1196,"ConnectOne Bancorp, Inc. | DEF 14A | 2025-04-1...",. Because the Annual Meeting is virtual and be...
3,d2231af2-9581-49fd-ac5b-271a658df8a7,3,712771_000143774925011656,preamble,,1173,"ConnectOne Bancorp, Inc. | DEF 14A | 2025-04-1...",. The webcast starts at 9:15 a.m. We encourage...
4,b1c24d1d-7d42-402e-974f-86327eee6d97,4,712771_000143774925011656,preamble,,1175,"ConnectOne Bancorp, Inc. | DEF 14A | 2025-04-1...",. The Board’s recommendation is set forth toge...


## M0 Assertions — Evidence Gate


In [14]:
assert len(blocks) > 50, f"Expected >50 blocks, got {len(blocks)}"
assert len(tables) >= 5, f"Expected >=5 tables in CNOB DEF 14A, got {len(tables)}"
assert len(headings) >= 10, f"Expected >=10 section headings, got {len(headings)}"
assert len(chunks) > 50, f"Expected >50 chunks, got {len(chunks)}"
assert all(c.citation_string for c in chunks), "Some chunks missing citation_string"
assert all(c.token_count > 0 for c in chunks), "Some chunks have zero token count"
# Every chunk's text must map back to source HTML (exact or whitespace-normalized)
import re
from bs4 import BeautifulSoup
normalized_raw_html = " ".join(raw_html.split())
normalized_visible_html = " ".join(BeautifulSoup(raw_html, "lxml").get_text(" ", strip=True).split())
for c in chunks:
    if c.table_json is not None:
        continue
    normalized_chunk = " ".join(c.text.split())
    candidate_texts = [normalized_chunk]
    # Image chunks contain synthetic [IMAGE:...] tokens not present in raw HTML.
    stripped = re.sub(r"^\[IMAGE:[^\]]+\]\s*", "", normalized_chunk).strip()
    if stripped and stripped != normalized_chunk:
        candidate_texts.append(stripped)
    assert any(t in normalized_raw_html or t in normalized_visible_html for t in candidate_texts), (
        f"Chunk text not traceable to source HTML: {c.text[:60]}"
    )
print("✓ All M0 assertions passed")


/var/folders/zf/tp4v74hs32z6n9m3gsshdth80000gn/T/ipykernel_73020/1736468387.py:11: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  normalized_visible_html = " ".join(BeautifulSoup(raw_html, "lxml").get_text(" ", strip=True).split())


✓ All M0 assertions passed
